# PART 03 · 실습예제 2 — LangChain/LangGraph 기반 비디오 모션 에이전트 + Colab
**video_LanGraph_Gemini_agent.ipynb** · LLM: **Google Gemini Flash**

업로드한 mp4 영상(최대 6개, 이름 자유)을 LangGraph **`Send` 병렬 분기**로 동시에 분석한다.
프레임 차분 스코어 → 이동평균(N=5) > TH(0.035) 판정 → 영상별 요약 → **Google Gemini Flash** 해석 (모의 LLM 없음, 실제 API 호출).

```
START ─(Send × 6, 병렬)→ vision_encode ─(operator.add 로 모음)→ temporal_aggregate → llm_answer → END
```

| 클린 아키텍처 계층 | 이 노트북 | 외부 의존 |
|---|---|---|
| 도메인 | `MotionParams`, `MotionReport`, `judge_motion()` (3단계) | 없음 |
| 포트 | `VideoScorer`, `ChatLLM` (`Protocol`, 4단계) | 없음 |
| 어댑터 | `OpenCVFrameDiffScorer`, `init_chat_model` → Google Gemini Flash (4·5단계) | OpenCV, LangChain |
| 프레임워크 | `build_graph()` (6단계) | LangGraph |

**실행 순서**: 런타임 → 모두 실행. 6-1·6-2단계에서 그래프 구조와 실행 흐름을 영상·API 없이 먼저 검증한다. 1단계에서 API Key 를 입력하고, 2단계 업로드 창에서 mp4 를 최대 6개 고르면 끝까지 진행된다.

**준비물**: Google Gemini Flash API Key (실제 LLM을 호출하므로 키가 반드시 필요하다)

## 0단계 · 패키지 설치

In [ ]:
%pip install -qU langgraph langchain langchain-google-genai

## 1단계 · API Key 입력
아래 `GOOGLE_API_KEY = "AIza*****"` 란에 키를 직접 붙여넣거나, 그대로 두고 실행하면 **입력창**이 뜬다 (입력값은 화면에 보이지 않음).
키를 입력할 때까지 입력창이 다시 뜬다. Gemini 키는 `sk-` 가 아니라 `AIza` 로 시작한다. 발급: https://aistudio.google.com/apikey
> 강의 자료를 공유할 때는 키를 붙여넣은 줄을 다시 `"AIza*****"` 로 되돌린다.

In [ ]:
import os
from getpass import getpass

GOOGLE_API_KEY = "AIza*****"   # ← 여기에 키를 붙여넣는다. 그대로 두면 아래 입력창이 뜬다

if "*" in GOOGLE_API_KEY:                                       # 자리표시자 그대로면
    GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")              # ① 이미 등록된 환경변수가 있으면 쓰고
    while not GOOGLE_API_KEY:                                    # ② 없으면 입력창 (빈 값이면 다시 묻는다)
        GOOGLE_API_KEY = getpass("Google Gemini Flash API Key 입력 (AIza*****): ").strip()
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY                          # LangChain 이 이 환경변수에서 키를 읽는다
if not GOOGLE_API_KEY.startswith("AIza"):
    print("⚠ 키가 'AIza' 로 시작하지 않습니다. 올바른 키인지 확인하세요.")
print("API Key 등록:", GOOGLE_API_KEY[:4] + "*****")

## 2단계 · 동영상 업로드 (최대 6개)
셀을 실행하면 Colab **업로드 창**(`파일 선택` 버튼)이 뜬다. mp4 를 이름 상관없이 **최대 6개**까지 한 번에 고른다.
- 파일을 고를 때까지 셀이 **기다리므로** "런타임 → 모두 실행" 으로 실행해도 된다.
- 6개를 넘게 고르면 앞의 6개만 등록한다. 이미 올린 영상이 있으면 업로드 창을 건너뛴다 (다시 올리려면 `REUPLOAD = True`).
- Colab 이 아닌 환경에서는 `VIDEO_DIR` 폴더의 mp4 를 최대 6개까지 쓴다.

In [ ]:
from pathlib import Path

MAX_VIDEOS = 6
REUPLOAD = False                                         # True: 이미 올린 영상이 있어도 업로드 창을 다시 띄운다
WORK_DIR = Path("videos")
WORK_DIR.mkdir(exist_ok=True)


def upload_videos() -> None:
    """Colab 업로드 창 → videos/원래이름.mp4 로 저장 (최대 6개). 파일을 고를 때까지 실행이 멈춘다."""
    from google.colab import files  # Colab 에서만 존재 (없으면 ImportError)
    for name in list(WORK_DIR.glob("*.mp4")):
        name.unlink()                                    # 새로 올리면 이전 영상은 지운다
    uploaded = files.upload()                            # {파일이름: 바이트}
    mp4s = [(n, d) for n, d in uploaded.items() if n.lower().endswith(".mp4")]
    for n in uploaded:
        Path(n).unlink(missing_ok=True)                  # files.upload 가 /content 에 남긴 원본 정리
    if len(mp4s) > MAX_VIDEOS:
        print(f"⚠ {len(mp4s)}개 중 앞의 {MAX_VIDEOS}개만 등록합니다")
    for name, data in mp4s[:MAX_VIDEOS]:
        (WORK_DIR / Path(name).name).write_bytes(data)
        print(f"✅ {Path(name).name} ({len(data) / 1e6:.1f} MB)")


def registered_videos() -> list[str]:
    """등록된 영상 목록 (videos/ → 없으면 VIDEO_DIR), 최대 6개."""
    for d in (WORK_DIR, Path(os.environ.get("VIDEO_DIR", WORK_DIR))):
        if found := sorted(str(p) for p in d.glob("*.mp4")):
            return found[:MAX_VIDEOS]
    return []


try:
    if REUPLOAD or not registered_videos():
        upload_videos()
except ImportError:                                      # Colab 이 아니면 업로드 창 없이 VIDEO_DIR 사용
    pass
print("등록된 영상:", [Path(v).name for v in registered_videos()])

## 3단계 · 도메인 — 판정 규칙 (순수 파이썬)
OpenCV·LangChain 을 모르는 계층이다. 숫자 리스트만으로 테스트할 수 있다.
- **이동평균**: 최근 N개 스코어 평균 (`deque(maxlen=N)`) — 1~2프레임짜리 잡음(그림자·조명 깜빡임)을 거른다
- **구간**: 평균 > TH 인 프레임이 이어진 범위 → `(시작초, 끝초)`

In [ ]:
from collections import deque
from dataclasses import dataclass


@dataclass(frozen=True)
class MotionParams:
    window: int = 5             # 이동평균 윈도우 N
    threshold: float = 0.035    # 판정 임계값 TH (바뀐 픽셀 비율)
    pixel_threshold: int = 25   # 픽셀이 '바뀌었다'고 볼 밝기 차이


@dataclass(frozen=True)
class MotionReport:
    video: str
    fps: float
    scores: list[float]         # 프레임별 차분 스코어 (원값)
    moving_avg: list[float]     # 이동평균
    raw_segments: list[tuple[float, float]]   # 원값 > TH 구간
    segments: list[tuple[float, float]]       # 이동평균 > TH 구간 (최종 판정)

    def line(self) -> str:
        """LLM 에 넣을 한 줄 요약."""
        def fmt(segs):
            return ", ".join(f"{a:.1f}~{b:.1f}s" for a, b in segs) or "없음"
        return (f"{self.video}: 최대 스코어 {max(self.scores):.3f} | 원값 {len(self.raw_segments)}구간 | "
                f"이동평균 {len(self.segments)}구간 {fmt(self.segments)}")


def moving_average(scores: list[float], n: int) -> list[float]:
    """최근 n개 스코어 평균. 시작 부분은 모인 만큼만 평균낸다."""
    buf: deque[float] = deque(maxlen=n)          # 오래된 값은 자동으로 밀려난다
    out = []
    for s in scores:
        buf.append(s)
        out.append(sum(buf) / len(buf))
    return out


def to_segments(flags: list[bool], fps: float) -> list[tuple[float, float]]:
    """연속된 True 구간 → [(시작초, 끝초), ...]"""
    segs, start = [], None
    for i, f in enumerate([*flags, False]):      # 끝에 False 를 붙여 마지막 구간도 닫는다
        if f and start is None:
            start = i
        elif not f and start is not None:
            segs.append((start / fps, (i - 1) / fps))
            start = None
    return segs


def judge_motion(video: str, fps: float, scores: list[float], p: MotionParams) -> MotionReport:
    ma = moving_average(scores, p.window)
    return MotionReport(video, fps, scores, ma,
                        raw_segments=to_segments([s > p.threshold for s in scores], fps),
                        segments=to_segments([a > p.threshold for a in ma], fps))


# 빠른 확인: 1프레임 잡음(0.1)은 원값으로는 구간이지만 이동평균에서는 걸러진다 → 원값 2구간, 이동평균 1구간
print(judge_motion("test", 10, [0, 0, 0, 0.1, 0, 0, 0.06, 0.06, 0.06, 0.06, 0], MotionParams()).line())

## 4단계 · 포트와 어댑터
- **포트**(`Protocol`): 유스케이스·그래프는 이 인터페이스에만 의존한다 (의존성 역전).
- `OpenCVFrameDiffScorer`: 회색조 → `absdiff` → 픽셀 임계 25 → **바뀐 픽셀 비율**. 실습예제 1·`motion_check.py` 와 같은 식이라 같은 숫자가 나온다.
- `ChatLLM` 포트는 LangChain 채팅 모델의 `invoke(messages)` 계약 그대로다 → `ChatOpenAI`·`ChatGoogleGenerativeAI` 가 별도 어댑터 없이 바로 꽂힌다.

In [ ]:
from typing import Protocol

import cv2
from langchain_core.messages import AIMessage


class VideoScorer(Protocol):
    def score(self, path: str) -> tuple[float, list[float]]: ...      # → (fps, 프레임별 스코어)


class ChatLLM(Protocol):
    def invoke(self, messages: list) -> AIMessage: ...                 # LangChain 채팅 모델 인터페이스


class OpenCVFrameDiffScorer:
    def __init__(self, pixel_threshold: int = 25):
        self.pixel_threshold = pixel_threshold

    def score(self, path: str) -> tuple[float, list[float]]:
        cap = cv2.VideoCapture(path)
        if not cap.isOpened():
            raise FileNotFoundError(path)
        fps, prev, scores = cap.get(cv2.CAP_PROP_FPS) or 10.0, None, []
        while (frame := cap.read()[1]) is not None:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            if prev is None:
                scores.append(0.0)                                   # 첫 프레임은 비교 대상이 없다
            else:
                _, mask = cv2.threshold(cv2.absdiff(prev, gray), self.pixel_threshold, 255, cv2.THRESH_BINARY)
                scores.append(cv2.countNonZero(mask) / mask.size)    # 바뀐 픽셀 비율 (0~1)
            prev = gray
        cap.release()
        return fps, scores

## 5단계 · LLM 객체 — Google Gemini Flash
LangChain 공식 문서의 `init_chat_model("공급자:모델")` 로 실제 LLM 객체를 만든다. 키는 1단계에서 넣은 `GOOGLE_API_KEY` 환경변수에서 읽는다.
Gemini-Flash 계열은 이름만 바꾸면 된다. 예: `google_genai:gemini-2.5-flash-lite`

In [ ]:
from langchain.chat_models import init_chat_model

MODEL = "google_genai:gemini-2.5-flash"


def make_llm() -> ChatLLM:
    """Google Gemini Flash 채팅 모델 객체 (어댑터 생성은 이 함수 한 곳에서만)."""
    return init_chat_model(MODEL, temperature=0)       # 키는 GOOGLE_API_KEY 환경변수에서 자동으로 읽는다


llm = make_llm()
print("LLM:", MODEL, "→", type(llm).__name__)

## 6단계 · LangGraph 조립 (공식 문서의 Map-Reduce · `Send` 패턴)
- `fan_out`: 영상마다 `Send("vision_encode", {...})` 를 만든다 → **같은 단계에서 병렬 실행**
- `vision_encode`: 영상 1개의 차분 스코어 + 이동평균 판정 → `reports` 에 추가 (`operator.add` 리듀서가 6개를 모은다)
- `temporal_aggregate`: 6개 판정을 요약 문자열로 / `llm_answer`: LLM 해석
- 의존 객체(scorer·llm·params)는 `build_graph()` 인자로 주입한다 → LLM·지표를 바꿔도 그래프 코드는 그대로 (개방-폐쇄)

In [ ]:
import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Send


class AgentState(TypedDict):
    videos: list[str]
    question: str
    reports: Annotated[list[MotionReport], operator.add]   # 병렬 분기 결과를 이어 붙인다
    summary: str
    answer: str


def build_graph(scorer: VideoScorer, llm: ChatLLM, params: MotionParams | None = None):
    """구성 루트: 의존 객체를 주입받아 그래프를 만든다."""
    params = params or MotionParams()

    def fan_out(state: AgentState) -> list[Send]:
        return [Send("vision_encode", {"video_path": v}) for v in state["videos"]]

    def vision_encode(state: dict) -> dict:
        fps, scores = scorer.score(state["video_path"])
        return {"reports": [judge_motion(Path(state["video_path"]).name, fps, scores, params)]}

    def temporal_aggregate(state: AgentState) -> dict:
        reports = sorted(state["reports"], key=lambda r: r.video)    # 병렬이라 도착 순서가 섞인다
        return {"summary": "\n".join(r.line() for r in reports)}

    def llm_answer(state: AgentState) -> dict:
        messages = [("system", "너는 CCTV 모션 분석 결과를 해석하는 관제 보조자다. 한국어로 간결하게 답한다."),
                    ("human", f"[판정 요약]\n{state['summary']}\n\n[질문]\n{state['question']}")]
        try:
            return {"answer": llm.invoke(messages).content}
        except Exception as e:                       # 잘못된 키·모델 이름: 분석 결과는 살리고 원인만 알린다
            return {"answer": f"[LLM 호출 실패 — API Key·모델 이름을 확인하세요] {type(e).__name__}: {str(e)[:150]}"}

    g = StateGraph(AgentState)
    g.add_node("vision_encode", vision_encode)
    g.add_node("temporal_aggregate", temporal_aggregate)
    g.add_node("llm_answer", llm_answer)
    g.add_conditional_edges(START, fan_out, ["vision_encode"])
    g.add_edge("vision_encode", "temporal_aggregate")
    g.add_edge("temporal_aggregate", "llm_answer")
    g.add_edge("llm_answer", END)
    return g.compile()


graph = build_graph(OpenCVFrameDiffScorer(), llm)

## 6-1단계 · 그래프 생성 검증 ① — 구조 확인 (노드·엣지·다이어그램)
6단계의 `graph = build_graph(...)` 는 **출력이 없어서** 그래프가 제대로 만들어졌는지 눈으로 볼 수 없다. 7단계에서 실제 영상·API 를 쓰기 전에 여기서 확인한다.
- `graph.get_graph()` 로 컴파일된 그래프의 **노드·엣지 목록**을 꺼내 설계도(`START → vision_encode → temporal_aggregate → llm_answer → END`)와 같은지 `assert` 로 검사
- `START → vision_encode` 가 **조건부 엣지**(`Send` 분기)인지 확인
- Mermaid 다이어그램을 그림(PNG)으로 표시 — 인터넷이 막혀 PNG 를 못 그리면 Mermaid 텍스트를 대신 출력

In [ ]:
from IPython.display import Image, display

g_view = graph.get_graph()                                     # 컴파일된 그래프의 구조(노드·엣지) 객체
nodes = set(g_view.nodes)
edges = {(e.source, e.target, e.conditional) for e in g_view.edges}

print("노드:", sorted(nodes))
print("엣지:")
for s, t, cond in sorted(edges):
    print(f"  {s:>18} → {t:<18} {'(조건부 · Send 병렬 분기)' if cond else ''}")

# 설계도와 같은지 검사 — 하나라도 다르면 AssertionError 로 7단계 전에 멈춘다
EXPECTED_NODES = {"__start__", "vision_encode", "temporal_aggregate", "llm_answer", "__end__"}
EXPECTED_EDGES = {("__start__", "vision_encode", True),               # fan_out → Send × N
                  ("vision_encode", "temporal_aggregate", False),
                  ("temporal_aggregate", "llm_answer", False),
                  ("llm_answer", "__end__", False)}
assert nodes == EXPECTED_NODES, f"노드 불일치: {nodes ^ EXPECTED_NODES}"
assert edges == EXPECTED_EDGES, f"엣지 불일치: {edges ^ EXPECTED_EDGES}"
print("\n✅ 노드 5개 · 엣지 4개가 설계도와 일치 (START→vision_encode 는 Send 조건부 엣지)")

try:
    display(Image(g_view.draw_mermaid_png()))                  # mermaid.ink 로 PNG 렌더링 (인터넷 필요)
except Exception as e:
    print(f"\n(PNG 렌더링 생략 — {type(e).__name__}) Mermaid 텍스트를 https://mermaid.live 에 붙여넣으면 그림으로 볼 수 있다:\n")
    print(g_view.draw_mermaid())

## 6-2단계 · 그래프 생성 검증 ② — 모의 실행 (영상·API 없이)
구조가 맞아도 **실행 흐름**이 맞는지는 돌려 봐야 안다. 포트(`VideoScorer`·`ChatLLM`)에 맞춘 **검증용 가짜 객체**를 `build_graph()` 에 주입해 같은 그래프 코드를 실행한다 (의존성 역전 덕분에 그래프 코드는 한 줄도 안 바뀐다).
- `FakeScorer`: 영상 파일을 열지 않고 정해진 스코어를 돌려준다 → 3단계 판정 규칙의 기대 구간과 비교
- `FakeLLM`: API 를 부르지 않고 받은 요약을 되돌려준다 → `llm_answer` 에 요약이 제대로 전달되는지 확인
- `stream_mode="updates"` 로 단계(super-step)별 실행 노드를 출력 → `vision_encode` 6개가 **같은 단계에서 병렬** 실행되고, `operator.add` 리듀서가 결과 6개를 모으는지 확인
- `BrokenLLM`: 호출이 실패해도 분석 결과(`summary`)는 남고 `answer` 에 원인만 표시되는지 확인

> 실제 Gemini 호출·영상 분석은 하지 않으므로 비용·시간이 들지 않는다. 모두 ✅ 이면 7단계로 진행한다.

In [ ]:
import time


class FakeScorer:
    """VideoScorer 포트 구현 (검증용): 파일 이름 → 미리 정한 스코어. fps=10 → 프레임 i 는 i/10 초."""
    PATTERNS = {
        "still.mp4":  [0.0] * 20,                                            # 움직임 없음
        "noise.mp4":  [0.0] * 5 + [0.1] + [0.0] * 14,                        # 1프레임 잡음 → 이동평균이 거른다
        "walk.mp4":   [0.0] * 5 + [0.06] * 6 + [0.0] * 9,                    # 0.5~1.5초 지속 움직임
        "twice.mp4":  [0.06] * 4 + [0.0] * 8 + [0.06] * 4 + [0.0] * 4,       # 움직임 2번
        "end.mp4":    [0.0] * 14 + [0.06] * 6,                               # 끝까지 이어지는 움직임
        "door.mp4":   [0.0] * 3 + [0.2] * 2 + [0.0] * 15,                    # 짧지만 강한 변화
    }

    DELAY = 0.5                                                              # 영상 1개 분석에 걸리는 척하는 시간

    def score(self, path: str) -> tuple[float, list[float]]:
        time.sleep(self.DELAY)
        return 10.0, self.PATTERNS[Path(path).name]


class FakeLLM:
    """ChatLLM 포트 구현 (검증용): API 대신 받은 메시지를 그대로 돌려준다."""
    def __init__(self):
        self.calls = []

    def invoke(self, messages: list) -> AIMessage:
        self.calls.append(messages)
        return AIMessage(content="[FakeLLM 응답]\n" + messages[-1][1])


class BrokenLLM:
    """ChatLLM 포트 구현 (검증용): 잘못된 API Key 처럼 항상 실패한다."""
    def invoke(self, messages: list) -> AIMessage:
        raise PermissionError("API key not valid (검증용 가짜 오류)")


# 3단계 판정 규칙(N=5, TH=0.035)으로 손 계산한 기대 구간 (round 1자리)
DRY_EXPECTED = {"still.mp4": [], "noise.mp4": [], "walk.mp4": [(0.7, 1.2)],
                "twice.mp4": [(0.0, 0.5), (1.4, 1.7)], "end.mp4": [(1.6, 1.9)], "door.mp4": [(0.3, 0.8)]}
DRY_VIDEOS = [f"dry/{name}" for name in reversed(FakeScorer.PATTERNS)]   # 존재하지 않는 경로 → 파일을 안 여는지도 확인
DRY_INPUT = {"videos": DRY_VIDEOS, "question": "검증 질문", "reports": []}

fake_llm = FakeLLM()
dry_graph = build_graph(FakeScorer(), fake_llm)                # 6단계와 같은 함수, 의존 객체만 교체

# ① 실행 흐름: Send 분기마다 업데이트가 하나씩 나오고, 모두 끝난 뒤에 다음 노드로 간다
print("[실행 흐름 · stream_mode='updates']")
order = []
t0 = time.perf_counter()
for chunk in dry_graph.stream(DRY_INPUT, stream_mode="updates"):
    order += list(chunk)
    print(f"  {len(order)}: {list(chunk)[0]}")
elapsed = time.perf_counter() - t0
assert order == ["vision_encode"] * len(DRY_VIDEOS) + ["temporal_aggregate", "llm_answer"], order
print(f"✅ Send 분기 {len(DRY_VIDEOS)}개 → temporal_aggregate → llm_answer 순서로 실행")

# 순차 실행이면 0.5초 × 6 = 3초, 병렬이면 약 0.5초
serial = FakeScorer.DELAY * len(DRY_VIDEOS)
assert elapsed < serial * 0.6, f"병렬 실행이 아니다: {elapsed:.2f}초 (순차 {serial:.1f}초)"
print(f"✅ 병렬 실행: {elapsed:.2f}초 (순차였다면 {serial:.1f}초)")

# ② 최종 상태: 리듀서·요약·LLM 전달 확인
fake_llm.calls.clear()
dry = dry_graph.invoke(DRY_INPUT)
assert len(dry["reports"]) == len(DRY_VIDEOS), "operator.add 리듀서가 결과를 모두 모으지 못했다"
print(f"✅ operator.add 리듀서: 병렬 결과 {len(dry['reports'])}개 수집")

for r in sorted(dry["reports"], key=lambda r: r.video):
    got = [(round(a, 1), round(b, 1)) for a, b in r.segments]
    assert got == DRY_EXPECTED[r.video], f"{r.video}: 기대 {DRY_EXPECTED[r.video]}, 결과 {got}"
    print("✅", r.line())
noise = next(r for r in dry["reports"] if r.video == "noise.mp4")
assert len(noise.raw_segments) == 1 and not noise.segments, "1프레임 잡음이 이동평균에서 걸러지지 않았다"
print("✅ 이동평균 잡음 필터: noise.mp4 원값 1구간 → 이동평균 0구간")

lines = dry["summary"].splitlines()
assert lines == sorted(lines) and len(lines) == len(DRY_VIDEOS), "temporal_aggregate 요약이 정렬·완전하지 않다"
assert len(fake_llm.calls) == 1, "LLM 은 그래프 실행 1번에 1번만 호출돼야 한다"
assert dry["summary"] in dry["answer"] and "검증 질문" in dry["answer"], "llm_answer 에 요약·질문이 전달되지 않았다"
print("✅ temporal_aggregate: 영상 이름순 요약 / llm_answer: 실행당 1번 호출, 요약·질문 전달")

# ③ LLM 실패 처리: 분석 결과는 살고 answer 에 원인만 남는다
broken = build_graph(FakeScorer(), BrokenLLM()).invoke(DRY_INPUT)
assert broken["summary"] == dry["summary"] and broken["answer"].startswith("[LLM 호출 실패"), broken["answer"]
print("✅ LLM 실패 처리:", broken["answer"])

print("\n🎉 그래프 생성 검증 완료 — 7단계에서 실제 영상 + Google Gemini Flash 로 실행한다")

## 7단계 · 실행 — 영상 동시 분석 + Google Gemini Flash 해석
2단계에서 올린 영상(최대 6개)을 모아 그래프를 한 번 실행한다. LLM 은 영상별 요약을 모아 **한 번** 호출한다.

In [ ]:
import time

VIDEOS = registered_videos()
if not VIDEOS:                                           # 2단계에서 취소했다면 여기서 한 번 더 업로드 창을 띄운다
    upload_videos()
    VIDEOS = registered_videos()
if not VIDEOS:
    raise FileNotFoundError("등록된 mp4 영상이 없습니다. 2단계 셀을 다시 실행해 영상을 올리세요.")

QUESTION = "영상별 움직임 구간을 해석하고, 원값 대비 이동평균이 걸러낸 잡음과 가장 주의가 필요한 영상을 알려줘."
t0 = time.perf_counter()
result = graph.invoke({"videos": VIDEOS, "question": QUESTION, "reports": []})
print(f"영상 {len(VIDEOS)}개 동시 분석 {time.perf_counter() - t0:.2f}초\n")
print(result["summary"], "\n")
print(result["answer"])

## 8단계 · 기대값 확인 + 스코어 그래프
강의용 `sample1~6.mp4` 는 `videos/README.txt`·실습예제 1(Go) 의 이동평균 판정과 같으면 ✅, 다른 영상은 판정 결과만 보여 준다. (그래프 글자는 Colab 한글 글꼴 문제로 영어)

In [ ]:
import matplotlib.pyplot as plt

EXPECTED = {"sample1.mp4": [(4.5, 6.5)], "sample2.mp4": [(6.0, 8.0)], "sample3.mp4": [],
            "sample4.mp4": [(2.5, 4.5), (10.0, 12.0)], "sample5.mp4": [(2.5, 3.1), (6.3, 7.6)],
            "sample6.mp4": [(10.5, 12.5)]}
reports = sorted(result["reports"], key=lambda r: r.video)
for r in reports:
    got = [(round(a, 1), round(b, 1)) for a, b in r.segments]
    mark = "  " if r.video not in EXPECTED else ("✅" if got == EXPECTED[r.video] else f"❌ 기대 {EXPECTED[r.video]}")
    print(mark, r.line())

fig, axes = plt.subplots(2, 3, figsize=(15, 6), sharey=True)
for ax in axes.flat[len(reports):]:
    ax.axis("off")                                   # 영상이 6개 미만이면 남는 칸을 숨긴다
for ax, r in zip(axes.flat, reports, strict=False):
    t = [i / r.fps for i in range(len(r.scores))]
    ax.plot(t, r.scores, color="#9aa3b5", lw=1, label="raw score")
    ax.plot(t, r.moving_avg, color="#1f3a68", lw=2, label="moving avg (N=5)")
    ax.axhline(MotionParams().threshold, color="#d9534f", ls="--", lw=1, label="TH")
    for a, b in r.segments:
        ax.axvspan(a, b, color="#f28c28", alpha=0.25)
    ax.set_title(r.video)
axes.flat[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 정리 · SOLID 적용
| 원칙 | 이 노트북 |
|---|---|
| S 단일 책임 | 판정(`judge_motion`), 스코어(`OpenCVFrameDiffScorer`), LLM 생성(`make_llm`), 흐름(`build_graph`) 분리 |
| O 개방-폐쇄 | 새 지표·새 LLM 은 포트 구현 클래스를 **추가**해 `build_graph()` 에 넣으면 끝 |
| L 리스코프 치환 | `ChatOpenAI` ↔ `ChatGoogleGenerativeAI`: 같은 `invoke().content` 계약 → OpenAI·Gemini 노트북은 0·1·5단계 설정값만 다르다 |
| I 인터페이스 분리 | 포트 2개가 각각 메서드 1개 |
| D 의존성 역전 | 그래프 Node 는 `VideoScorer`·`ChatLLM` 포트에만 의존, 구체 객체는 6단계 마지막 줄에서 주입 |

참고한 공식 문서: LangGraph `Send` Map-Reduce (docs.langchain.com/oss/python/langgraph/graph-api), 리듀서 `Annotated[list, operator.add]`, LangChain `init_chat_model` (docs.langchain.com/oss/python/langchain/models)

---
## 유사 적용 인용 사례 (실제 프로젝트)
공개 보도자료·공식 페이지로 확인한 내용만 적었다. 공개 자료에 없는 항목은 **'공개 자료 미기재'** 로 표시했다.

| # | 프로젝트 (연도) | 주관 | 참가 기업·기관 | 수요처 | 이 노트북과의 연결 |
|---|---|---|---|---|---|
| 1 | **NVIDIA AI Blueprint for Video Search and Summarization** (2024.11) | NVIDIA | Accenture(AI Refinery 통합), Dell(NativeEdge), Lenovo, K2K, ITMAX(말레이시아), FPT(베트남) | 이탈리아 팔레르모시 교통 관리 (K2K 적용) | 다수 카메라 영상 분석 → 비전 요약 → LLM 질의응답 에이전트 |
| 2 | **Project Hafnia** (2025.06 유럽 확장) | Milestone Systems | NVIDIA(VLM·Cosmos 합성 영상), Nebius(EU GPU 클라우드) | 이탈리아 제노바시 (교통·도시 인프라) | 도시 CCTV 영상으로 학습한 VLM 으로 교통 상황 해석 |
| 3 | **KISA 지능형 CCTV 성능 시험인증** (2017.03 첫 인증) | 한국인터넷진흥원(KISA) | 첫 인증: 일리시스 IntelliVIX, KT 기가아이즈 VA (배회·침입·유기) / 이후 피아스페이스 6개 부문(배회·침입·쓰러짐·방화·싸움·유기, 2025.03 보도) | 지자체 CCTV 관제센터 등 (기업별 수요처는 공개 자료 미기재) | 이상행위 구간을 시간 범위로 탐지 — 이 노트북의 '구간 판정'과 같은 평가 단위 |
| 4 | **AI Hub 이상행동 CCTV 영상** 데이터 구축 (2019) | 마인즈랩 (AI Hub 데이터 페이지 기준) | ㈜마인즈앤컴퍼니, 수원시청 | 공개 데이터 (AI Hub, 과기정통부·NIA 운영) | 폭행·배회·침입·투기 등 12종 717시간 — 모션 판정 모델의 학습·평가 데이터 |
| 5 | **지능형 관제 서비스 CCTV 영상 데이터** 구축 (2024 초거대 AI 데이터 사업) | 과학기술정보통신부·NIA | 수행기관 한국딥러닝 (컨소시엄 명단은 공개 자료 미기재) | 지능형 CCTV 운영기관 (구체 기관명 미기재) | 침입·쓰러짐·싸움·인파 밀집 감지용 데이터 |
| 6 | **LangGraph 프로덕션 사례** (LangChain 공식 Case studies) | LangChain | Klarna, Uber, LinkedIn, Elastic, J.P. Morgan 등 | 각 기업 내부 서비스 | 비디오 사례는 아니지만, 이 노트북과 같은 StateGraph·병렬 분기 구조를 실서비스에 쓴 사례 |

출처: blogs.nvidia.com (2024-11-04 VSS Blueprint) · milestonesys.com 보도자료 (2025-06-12) · edaily (2017-03, KISA 첫 인증) · 인공지능신문 aitimes.kr (2025-03-17, 피아스페이스) · aihub.or.kr 데이터셋 171 · 전자신문 etnews (2024-12-03, 한국딥러닝) · docs.langchain.com/oss/python/langgraph/case-studies